# SAM 3D 稀疏点云提取与蒸馏加速教程

本教程演示如何：
1. 从图像中提取稀疏点云坐标和特征
2. 使用 Shortcut 蒸馏方法加速生成 (25步 → 4步)
3. 对比标准模式与蒸馏模式的性能和质量
4. 可视化稀疏点云和中间结果

## 背景

SAM 3D 使用两阶段生成流程：
- **阶段 1**: 稀疏结构生成 (Sparse Structure) - 生成 64³ 体素网格，提取占据体素坐标
- **阶段 2**: 结构化潜在生成 (Structured Latent) - 在稀疏坐标上生成详细特征

蒸馏加速基于 **Shortcut Models** (论文 Section 4 & C.4)，将推理步数从 25 减少到 4，同时保持生成质量。

## 1. 导入和模型加载

In [ ]:
# Copyright (c) Meta Platforms, Inc. and affiliates.

import os
import sys
import time
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 添加 notebook 目录到路径
sys.path.insert(0, os.getcwd())

from inference import (
    Inference, 
    load_image, 
    load_single_mask, 
    display_image
)

In [ ]:
# 设置路径
PATH = os.getcwd()
TAG = "hf"
config_path = f"{PATH}/../checkpoints/{TAG}/pipeline.yaml"

print(f"配置文件路径: {config_path}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 设备: {torch.cuda.get_device_name(0)}")
    print(f"GPU 显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 加载模型
# 注意：首次加载可能需要较长时间
print("正在加载模型...")
start_time = time.time()
inference = Inference(config_path, compile=False)
load_time = time.time() - start_time
print(f"模型加载完成，耗时: {load_time:.2f} 秒")

## 2. 基础流程：提取稀疏点云

### 2.1 加载输入图像和掩码

In [ ]:
# 加载示例图像
IMAGE_PATH = f"{PATH}/images/shutterstock_stylish_kidsroom_1640806567/image.png"
IMAGE_NAME = os.path.basename(os.path.dirname(IMAGE_PATH))

image = load_image(IMAGE_PATH)
mask = load_single_mask(os.path.dirname(IMAGE_PATH), index=14)

print(f"图像尺寸: {image.shape}")
print(f"掩码尺寸: {mask.shape}")
print(f"掩码像素数: {mask.sum()}")

display_image(image, masks=[mask])

### 2.2 理解稀疏点云生成流程

稀疏点云生成包含以下步骤：

1. **条件嵌入**: 使用 DINO 提取图像特征
2. **Flow Matching**: 在潜在空间中进行去噪
3. **VAE 解码**: 将潜在表示解码为 64³ 体素网格
4. **坐标提取**: 提取占据体素的坐标

```
输入图像 (H, W, 3)
    │
    ▼
DINO 特征提取
    │
    ▼
Flow Matching (潜在空间: 4096 x 8)
    │
    ▼
VAE 解码 (16³ → 64³ 体素网格)
    │
    ▼
稀疏点云坐标 (N, 3)
```

In [ ]:
# 访问底层管道以获取中间结果
pipeline = inference._pipeline

# 打印模型组件
print("=== 模型组件 ===")
for key in pipeline.models.keys():
    print(f"  - {key}")

print("\n=== 条件嵌入器 ===")
for key in pipeline.condition_embedders.keys():
    print(f"  - {key}")

### 2.3 标准模式运行 (25 步)

In [ ]:
# 标准模式：25 步推理
print("=== 标准模式 (25 步) ===")
print("参数: no_shortcut=True, CFG strength=7.0")

torch.cuda.reset_peak_memory_stats()
start_time = time.time()

# 标准推理
output_standard = inference(image, mask, seed=42)

standard_time = time.time() - start_time
standard_memory = torch.cuda.max_memory_allocated() / 1e9

print(f"推理时间: {standard_time:.2f} 秒")
print(f"峰值显存: {standard_memory:.2f} GB")
print(f"输出键: {list(output_standard.keys())}")

In [ ]:
# 提取稀疏点云信息
gs_standard = output_standard["gs"]

# 获取高斯点云坐标
xyz_standard = gs_standard.get_xyz.cpu().numpy()
print(f"稀疏点云坐标形状: {xyz_standard.shape}")
print(f"点云范围: [{xyz_standard.min():.3f}, {xyz_standard.max():.3f}]")
print(f"点云中心: [{xyz_standard.mean(axis=0)}]")

# 获取高斯特征
features_dc = gs_standard._features_dc.cpu().numpy()
print(f"\n特征维度:")
print(f"  - 位置 (xyz): {gs_standard._xyz.shape}")
print(f"  - 特征 DC: {features_dc.shape}")
print(f"  - 缩放: {gs_standard._scaling.shape}")
print(f"  - 旋转: {gs_standard._rotation.shape}")
print(f"  - 不透明度: {gs_standard._opacity.shape}")

## 3. 蒸馏加速方法

### 3.1 Shortcut 模型原理

基于论文 Section C.4，Shortcut 模型通过以下方式加速：

1. **混合训练目标**：
   - 75% 样本使用标准 Flow Matching (d=0)
   - 25% 样本使用 Self-Consistency Loss (d>0)

2. **推理时**：
   - 每步使用均匀步长 d = 1/steps
   - CFG 强度已蒸馏到模型中

3. **性能对比**：
   | 模式 | 步数 | CFG | 速度提升 |
   |------|------|-----|----------|
   | 标准 | 25 | 7.0 | 1x |
   | Shortcut | 4 | 0.0 | ~6x |
   | Shortcut | 8 | 0.0 | ~3x |

### 3.2 使用蒸馏模式运行

In [ ]:
# 蒸馏模式：通过管道直接访问
print("=== 蒸馏模式 (4 步) ===")
print("参数: no_shortcut=False, CFG strength=0 (蒸馏到模型中)")

# 准备输入
rgba_image = inference.merge_mask_to_rgba(image, mask)

torch.cuda.reset_peak_memory_stats()
start_time = time.time()

# 使用蒸馏模式运行
output_distill = pipeline.run(
    rgba_image,
    None,
    seed=42,
    stage1_only=False,
    with_mesh_postprocess=False,
    with_texture_baking=False,
    with_layout_postprocess=False,
    use_vertex_color=True,
    stage1_inference_steps=4,      # 4 步推理
    stage2_inference_steps=4,      # 4 步推理
    use_stage1_distillation=True,  # 启用 Stage 1 蒸馏
    use_stage2_distillation=True,  # 启用 Stage 2 蒸馏
)

distill_time = time.time() - start_time
distill_memory = torch.cuda.max_memory_allocated() / 1e9

print(f"推理时间: {distill_time:.2f} 秒")
print(f"峰值显存: {distill_memory:.2f} GB")

In [ ]:
# 提取蒸馏模式结果
# 注意: output["gs"] 已经是单个 Gaussian 对象，不需要 [0] 索引
gs_distill = output_distill["gs"]
xyz_distill = gs_distill.get_xyz.cpu().numpy()

print(f"蒸馏模式点云坐标形状: {xyz_distill.shape}")
print(f"点云范围: [{xyz_distill.min():.3f}, {xyz_distill.max():.3f}]")

### 3.3 不同步数的蒸馏模式对比

In [ ]:
# 测试不同步数的蒸馏模式
steps_to_test = [2, 4, 8, 12]
distill_results = {}

print("=== 不同步数的蒸馏模式对比 ===\n")

for steps in steps_to_test:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    start_time = time.time()
    
    output = pipeline.run(
        rgba_image,
        None,
        seed=42,
        stage1_only=False,
        with_mesh_postprocess=False,
        with_texture_baking=False,
        with_layout_postprocess=False,
        use_vertex_color=True,
        stage1_inference_steps=steps,
        stage2_inference_steps=steps,
        use_stage1_distillation=True,
        use_stage2_distillation=True,
    )
    
    elapsed = time.time() - start_time
    memory = torch.cuda.max_memory_allocated() / 1e9
    
    distill_results[steps] = {
        'time': elapsed,
        'memory': memory,
        'output': output,
        'num_points': output['gs'].get_xyz.shape[0]  # gs 已经是单个对象
    }
    
    print(f"步数: {steps:2d} | 时间: {elapsed:.2f}s | 显存: {memory:.2f}GB | 点数: {distill_results[steps]['num_points']}")

## 4. 对比分析

### 4.1 推理时间对比

In [ ]:
# 绘制时间对比图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 时间对比
steps_list = [25] + steps_to_test
times = [standard_time] + [distill_results[s]['time'] for s in steps_to_test]
colors = ['#2ecc71'] + ['#3498db'] * len(steps_to_test)

ax1 = axes[0]
bars = ax1.bar(range(len(steps_list)), times, color=colors)
ax1.set_xticks(range(len(steps_list)))
ax1.set_xticklabels([f'标准\n25步'] + [f'蒸馏\n{s}步' for s in steps_to_test])
ax1.set_ylabel('推理时间 (秒)', fontsize=12)
ax1.set_title('推理时间对比', fontsize=14)

# 添加数值标签
for bar, t in zip(bars, times):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
             f'{t:.2f}s', ha='center', va='bottom', fontsize=10)

# 加速比
ax2 = axes[1]
speedups = [standard_time / t for t in times]
bars2 = ax2.bar(range(len(steps_list)), speedups, color=colors)
ax2.set_xticks(range(len(steps_list)))
ax2.set_xticklabels([f'标准\n25步'] + [f'蒸馏\n{s}步' for s in steps_to_test])
ax2.set_ylabel('加速比', fontsize=12)
ax2.set_title('相对于标准模式的加速比', fontsize=14)
ax2.axhline(y=1, color='red', linestyle='--', label='基准线')

# 添加数值标签
for bar, s in zip(bars2, speedups):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
             f'{s:.1f}x', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(f"{PATH}/gaussians/single/performance_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\n加速效果总结:")
print(f"  标准 25 步: {standard_time:.2f}s (基准)")
for s in steps_to_test:
    print(f"  蒸馏 {s:2d} 步: {distill_results[s]['time']:.2f}s ({standard_time/distill_results[s]['time']:.1f}x 加速)")

### 4.2 点云质量对比

In [ ]:
# 计算点云统计信息
def compute_point_cloud_stats(xyz):
    """计算点云统计信息"""
    return {
        'num_points': len(xyz),
        'center': xyz.mean(axis=0),
        'std': xyz.std(axis=0),
        'range': xyz.max(axis=0) - xyz.min(axis=0),
    }

print("=== 点云统计对比 ===\n")

# 标准模式
stats_standard = compute_point_cloud_stats(xyz_standard)
print(f"标准模式 (25步):")
print(f"  点数: {stats_standard['num_points']}")
print(f"  中心: {stats_standard['center']}")
print(f"  标准差: {stats_standard['std']}")

print(f"\n蒸馏模式:")
for steps in steps_to_test:
    xyz = distill_results[steps]['output']['gs'].get_xyz.cpu().numpy()
    stats = compute_point_cloud_stats(xyz)
    print(f"  {steps}步 - 点数: {stats['num_points']}, 中心: {stats['center']}")

In [ ]:
# 计算 Chamfer Distance（近似）
def compute_chamfer_distance(xyz1, xyz2, sample_size=1000):
    """计算两个点云之间的 Chamfer Distance（采样近似）"""
    # 随机采样以加速计算
    idx1 = np.random.choice(len(xyz1), min(sample_size, len(xyz1)), replace=False)
    idx2 = np.random.choice(len(xyz2), min(sample_size, len(xyz2)), replace=False)
    
    pts1 = xyz1[idx1]
    pts2 = xyz2[idx2]
    
    # 计算双向最近邻距离
    from scipy.spatial import cKDTree
    tree1 = cKDTree(pts1)
    tree2 = cKDTree(pts2)
    
    dist1, _ = tree2.query(pts1)
    dist2, _ = tree1.query(pts2)
    
    return (dist1.mean() + dist2.mean()) / 2

# 尝试计算（需要 scipy）
try:
    print("\n=== Chamfer Distance 对比 (相对于标准模式) ===\n")
    for steps in steps_to_test:
        xyz = distill_results[steps]['output']['gs'].get_xyz.cpu().numpy()
        cd = compute_chamfer_distance(xyz_standard, xyz)
        print(f"  蒸馏 {steps}步: CD = {cd:.4f}")
except ImportError:
    print("需要 scipy 来计算 Chamfer Distance")

## 5. 可视化展示

### 5.1 3D 点云可视化

In [ ]:
# 3D 点云可视化函数
def visualize_point_cloud(ax, xyz, title, color='blue', alpha=0.6, s=1):
    """可视化点云"""
    ax.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2], c=color, alpha=alpha, s=s)
    ax.set_title(title)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    # 设置相等的比例
    max_range = np.array([xyz[:, 0].max()-xyz[:, 0].min(), 
                          xyz[:, 1].max()-xyz[:, 1].min(), 
                          xyz[:, 2].max()-xyz[:, 2].min()]).max() / 2.0
    mid_x = (xyz[:, 0].max()+xyz[:, 0].min()) * 0.5
    mid_y = (xyz[:, 1].max()+xyz[:, 1].min()) * 0.5
    mid_z = (xyz[:, 2].max()+xyz[:, 2].min()) * 0.5
    ax.set_xlim(mid_x - max_range, mid_x + max_range)
    ax.set_ylim(mid_y - max_range, mid_y + max_range)
    ax.set_zlim(mid_z - max_range, mid_z + max_range)

# 创建对比图
fig = plt.figure(figsize=(16, 12))

# 标准模式
ax1 = fig.add_subplot(2, 2, 1, projection='3d')
visualize_point_cloud(ax1, xyz_standard, '标准模式 (25步)', color='green')

# 蒸馏模式
colors = ['blue', 'red', 'orange', 'purple']
for i, steps in enumerate([2, 4, 8]):
    ax = fig.add_subplot(2, 2, i+2, projection='3d')
    xyz = distill_results[steps]['output']['gs'].get_xyz.cpu().numpy()
    visualize_point_cloud(ax, xyz, f'蒸馏模式 ({steps}步)', color=colors[i])

plt.tight_layout()
plt.savefig(f"{PATH}/gaussians/single/point_cloud_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

### 5.2 输出结果保存

In [ ]:
# 保存结果
output_dir = f"{PATH}/gaussians/single"
os.makedirs(output_dir, exist_ok=True)

# 保存标准模式结果
output_standard["gs"].save_ply(f"{output_dir}/{IMAGE_NAME}_standard.ply")
print(f"已保存: {output_dir}/{IMAGE_NAME}_standard.ply")

# 保存蒸馏模式结果
for steps in steps_to_test:
    gs = distill_results[steps]['output']['gs']
    gs.save_ply(f"{output_dir}/{IMAGE_NAME}_distill_{steps}steps.ply")
    print(f"已保存: {output_dir}/{IMAGE_NAME}_distill_{steps}steps.ply")

## 6. 高级用法

### 6.1 访问中间特征

In [ ]:
# 访问稀疏结构生成的中间结果
print("=== 访问底层模型组件 ===")

# 获取稀疏结构生成器
ss_generator = pipeline.models['ss_generator']
print(f"\n稀疏结构生成器类型: {type(ss_generator).__name__}")

# 检查是否支持 shortcut
has_shortcut = hasattr(ss_generator, 'no_shortcut')
print(f"支持 Shortcut: {has_shortcut}")

if has_shortcut:
    print(f"当前 no_shortcut 设置: {ss_generator.no_shortcut}")

# 获取 VAE 解码器
ss_decoder = pipeline.models['ss_decoder']
print(f"\nVAE 解码器类型: {type(ss_decoder).__name__}")

# 获取条件嵌入器
ss_condition_embedder = pipeline.condition_embedders['ss_condition_embedder']
print(f"条件嵌入器类型: {type(ss_condition_embedder).__name__}")

### 6.2 自定义推理参数

In [ ]:
# 自定义参数的完整示例
def run_custom_inference(
    image, 
    mask,
    stage1_steps=4,
    stage2_steps=4,
    use_distillation=True,
    seed=42
):
    """
    自定义推理函数
    
    参数:
        stage1_steps: 阶段1 (稀疏结构) 推理步数
        stage2_steps: 阶段2 (结构化潜在) 推理步数
        use_distillation: 是否使用蒸馏模式
        seed: 随机种子
    """
    rgba_image = inference.merge_mask_to_rgba(image, mask)
    
    return pipeline.run(
        rgba_image,
        None,
        seed=seed,
        stage1_only=False,
        with_mesh_postprocess=False,
        with_texture_baking=False,
        with_layout_postprocess=False,
        use_vertex_color=True,
        stage1_inference_steps=stage1_steps,
        stage2_inference_steps=stage2_steps,
        use_stage1_distillation=use_distillation,
        use_stage2_distillation=use_distillation,
    )

# 使用自定义函数
print("=== 使用自定义参数运行 ===")
output_custom = run_custom_inference(
    image, mask,
    stage1_steps=6,
    stage2_steps=6,
    use_distillation=True,
    seed=123
)
print(f"自定义推理完成，输出键: {list(output_custom.keys())}")

### 6.3 批量处理优化

In [ ]:
# 批量处理多个掩码
def batch_process_masks(image, mask_folder, indices, use_distillation=True, steps=4):
    """
    批量处理多个掩码
    
    参数:
        image: 输入图像
        mask_folder: 掩码文件夹路径
        indices: 掩码索引列表
        use_distillation: 是否使用蒸馏模式
        steps: 推理步数
    """
    results = []
    
    for idx in indices:
        mask = load_single_mask(mask_folder, index=idx)
        output = run_custom_inference(
            image, mask,
            stage1_steps=steps,
            stage2_steps=steps,
            use_distillation=use_distillation
        )
        results.append(output)
        
        # 清理显存
        torch.cuda.empty_cache()
    
    return results

# 示例：批量处理前3个掩码
print("=== 批量处理示例 ===")
print("跳过实际执行以节省时间...")
print("\n批量处理代码示例:")
print("""results = batch_process_masks(
    image, 
    os.path.dirname(IMAGE_PATH), 
    indices=[0, 1, 2],
    use_distillation=True,
    steps=4
)""")

## 7. 总结

### 关键要点

1. **稀疏点云生成流程**:
   - 输入图像 → DINO 特征 → Flow Matching → VAE 解码 → 稀疏点云
   - 输出: 64³ 体素网格中的占据坐标

2. **Shortcut 蒸馏加速**:
   - 启用 `use_stage1_distillation=True` 和 `use_stage2_distillation=True`
   - 推理步数从 25 减少到 4-8 步
   - CFG 强度已蒸馏到模型中，无需额外设置

3. **性能对比**:
   - 标准 25 步: 基准性能
   - 蒸馏 4 步: ~6x 加速，质量略有下降
   - 蒸馏 8 步: ~3x 加速，质量接近标准模式

4. **推荐设置**:
   - 快速预览: `steps=4, use_distillation=True`
   - 平衡模式: `steps=8, use_distillation=True`
   - 高质量: `steps=25, use_distillation=False` (标准模式)

In [ ]:
# 最终性能总结表
print("\n" + "="*60)
print("性能总结")
print("="*60)
print(f"{'模式':<15} {'步数':<8} {'时间(s)':<10} {'加速比':<10} {'推荐场景'}")
print("-"*60)
print(f"{'标准模式':<15} {25:<8} {standard_time:<10.2f} {'1.0x':<10} {'高质量输出'}")
for s in steps_to_test:
    speedup = standard_time / distill_results[s]['time']
    use_case = '快速预览' if s <= 4 else ('平衡模式' if s <= 8 else '高质量')
    print(f"{'蒸馏模式':<15} {s:<8} {distill_results[s]['time']:<10.2f} {speedup:<10.1f}x {use_case}")
print("="*60)